In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
train = pd.read_csv("../data/train.csv")
validation = pd.read_csv("../data/validation.csv")
test = pd.read_csv("../data/test.csv")

In [3]:
train.head()

,text
0,"First Citizen:\nBefore we proceed any further,..."


In [9]:
train.iloc[0, 0][:1000]

"First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.\n\nAll:\nWe know't, we know't.\n\nFirst Citizen:\nLet us kill him, and we'll have corn at our own price.\nIs't a verdict?\n\nAll:\nNo more talking on't; let it be done: away, away!\n\nSecond Citizen:\nOne word, good citizens.\n\nFirst Citizen:\nWe are accounted poor citizens, the patricians good.\nWhat authority surfeits on would relieve us: if they\nwould yield us but the superfluity, while it were\nwholesome, we might guess they relieved us humanely;\nbut they think we are too dear: the leanness that\nafflicts us, the object of our misery, is as an\ninventory to particularise their abundance; our\nsufferance is a gain to them Let us revenge this with\nour pikes, ere we become rakes: for the gods know I\nspeak this in hunger 

In [5]:
train.shape

(1, 1)

In [10]:
train.iloc[0,0][:500]

"First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.\n\nAll:\nWe know't, we know't.\n\nFirst Citizen:\nLet us kill him, and we'll have corn at our own price.\nIs't a verdict?\n\nAll:\nNo more talking on't; let it be done: away, away!\n\nSecond Citizen:\nOne word, good citizens.\n\nFirst Citizen:\nWe are accounted poor"

In [6]:
train.columns

Index(['text'], dtype='str')

In [7]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    1 non-null      str  
dtypes: str(1)
memory usage: 140.0 bytes


In [8]:
train.isnull().sum()

text    0
dtype: int64

In [11]:
text = train.loc[0, "text"]

print(text[:500])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [12]:
print("Number of characters:", len(text))

Number of characters: 1003854


In [13]:
print("Number of unique characters:", len(set(text)))

Number of unique characters: 65


In [14]:
print(sorted(set(text)))

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [15]:
chars = sorted(set(text))

print(chars)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [16]:
print("Vocabulary Size:", len(chars))

Vocabulary Size: 65


In [17]:
char_to_idx = {
    char: idx
    for idx, char in enumerate(chars)
}

In [18]:
idx_to_char = {
    idx: char
    for idx, char in enumerate(chars)
}

In [19]:
list(char_to_idx.items())[:20]

[('\n', 0),
 (' ', 1),
 ('!', 2),
 ('$', 3),
 ('&', 4),
 ("'", 5),
 (',', 6),
 ('-', 7),
 ('.', 8),
 ('3', 9),
 (':', 10),
 (';', 11),
 ('?', 12),
 ('A', 13),
 ('B', 14),
 ('C', 15),
 ('D', 16),
 ('E', 17),
 ('F', 18),
 ('G', 19)]

In [20]:
encoded_text = [
    char_to_idx[c]
    for c in text
]

In [21]:
encoded_text[:100]

[18,
 47,
 56,
 57,
 58,
 1,
 15,
 47,
 58,
 47,
 64,
 43,
 52,
 10,
 0,
 14,
 43,
 44,
 53,
 56,
 43,
 1,
 61,
 43,
 1,
 54,
 56,
 53,
 41,
 43,
 43,
 42,
 1,
 39,
 52,
 63,
 1,
 44,
 59,
 56,
 58,
 46,
 43,
 56,
 6,
 1,
 46,
 43,
 39,
 56,
 1,
 51,
 43,
 1,
 57,
 54,
 43,
 39,
 49,
 8,
 0,
 0,
 13,
 50,
 50,
 10,
 0,
 31,
 54,
 43,
 39,
 49,
 6,
 1,
 57,
 54,
 43,
 39,
 49,
 8,
 0,
 0,
 18,
 47,
 56,
 57,
 58,
 1,
 15,
 47,
 58,
 47,
 64,
 43,
 52,
 10,
 0,
 37,
 53,
 59]

In [22]:
seq_length = 100

In [23]:
input_sequences = []
target_chars = []

for i in range(len(encoded_text) - seq_length):
    input_sequences.append(encoded_text[i:i + seq_length])
    target_chars.append(encoded_text[i + seq_length])

In [24]:
X = np.array(input_sequences)
y = np.array(target_chars)

In [25]:
print("Input shape:", X.shape)
print("Target shape:", y.shape)

Input shape: (1003754, 100)
Target shape: (1003754,)


In [26]:
X = X[:100000]
y = y[:100000]

print(X.shape)
print(y.shape)

(100000, 100)
(100000,)


In [33]:
model = Sequential([
    Input(shape=(seq_length,)),

    Embedding(
        input_dim=len(chars),
        output_dim=64
    ),

    LSTM(128),

    Dense(
        len(chars),
        activation="softmax"
    )
])

In [35]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 100, 64)        │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 65)             │         8,385 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 111,361 (435.00 KB)

 Trainable params: 111,361 (435.00 KB)

 Non-trainable params: 0 (0.00 B)

In [36]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [37]:
history = model.fit(
    X,
    y,
    epochs=5,
    batch_size=64,
    validation_split=0.1
)

Epoch 1/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 101s 70ms/step - accuracy: 0.3163 - loss: 2.4692 - val_accuracy: 0.3804 - val_loss: 2.1429
Epoch 2/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 91s 64ms/step - accuracy: 0.4059 - loss: 2.0639 - val_accuracy: 0.4295 - val_loss: 1.9604
Epoch 3/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 94s 67ms/step - accuracy: 0.4400 - loss: 1.9201 - val_accuracy: 0.4546 - val_loss: 1.8504
Epoch 4/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 84s 60ms/step - accuracy: 0.4626 - loss: 1.8254 - val_accuracy: 0.4762 - val_loss: 1.7828
Epoch 5/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 88s 62ms/step - accuracy: 0.4807 - loss: 1.7548 - val_accuracy: 0.4879 - val_loss: 1.7312


In [38]:
model.save("../models/character_rnn.keras")

print("Model saved successfully🤓")

Model saved successfully🤓


In [41]:
def generate_text(model, seed_text, length=500, temperature=0.8):
    generated = seed_text

    for _ in range(length):

        # Encode the last seq_length characters
        encoded = [char_to_idx.get(c, 0) for c in generated[-seq_length:]]

        encoded = pad_sequences(
            [encoded],
            maxlen=seq_length,
            truncating="pre"
        )

        # Predict probabilities
        prediction = model.predict(encoded, verbose=0)[0]

        # Apply temperature
        prediction = np.log(prediction + 1e-8) / temperature
        prediction = np.exp(prediction)
        prediction = prediction / np.sum(prediction)

        # Sample next character
        next_index = np.random.choice(len(chars), p=prediction)

        # Convert index back to character
        next_char = idx_to_char[next_index]

        # Append character
        generated += next_char

    return generated

In [42]:
generated_text = generate_text(
    model,
    seed_text="First Citizen:",
    length=500,
    temperature=0.8
)

print(generated_text)

First Citizen:
Were hand
Sem you, Martson frorll't,
Where forten my good grade, fere in must there then gitel
Thour that do recest 't of you seat anderd; in you no
Hotht you marted of me froth late oftry hor graties,
Iur the patss ellile you are my arp,
And coscone: gereard to him may your in hall manown
He he inst mone core of of co
cike aghold ene you suart will our an dighteth
The recipulide ut will lided and the giter us Cole
I war sor gooctred of me cangly fore, meneran,
Ture dome be he so wen outels, no


In [45]:
with open("../outputs/generated_text.txt", "w") as f:
    f.write(generated_text)

print("Generated text saved👍🏻")

Generated text saved👍🏻
